<a href="https://colab.research.google.com/github/shaoran713-dotcom/Google-colab/blob/main/%E7%BE%8E%E8%82%A1%E5%9D%87%E5%80%BC%E5%9B%9E%E6%AD%B8%E6%A9%9F%E7%8E%87%E5%88%86%E6%9E%90_V6_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 美股均值回歸機率分析 V6.3
# ============================================
# 🛡️ 美股均值回歸機率分析 V6.3 - 核心運算引擎
# ============================================
import yfinance as yf
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
from datetime import datetime

# --- 技術指標模組 ---
def calculate_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).ewm(alpha=1/period, adjust=False).mean()
    loss = (-delta.where(delta < 0, 0)).ewm(alpha=1/period, adjust=False).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def calculate_atr(high, low, close, period=14):
    tr = pd.concat([
        high - low,
        np.abs(high - close.shift()),
        np.abs(low - close.shift())
    ], axis=1).max(axis=1)
    return tr.rolling(window=period).mean()

# --- 核心引擎 ---
def run_ultimate_v6_3(configs):
    market_tickers = ['ES=F', 'VIXY', 'VIXM']

    try:
        # 1. 環境掃描
        m_data = yf.download(market_tickers, period="5d", progress=False)['Close'].ffill()
        es_now = m_data['ES=F'].iloc[-1]
        es_prev = m_data['ES=F'].iloc[-2] if len(m_data) > 1 else es_now
        futures_change_pct = ((es_now - es_prev) / es_prev) * 100

        vix_ratio = (m_data['VIXY'] / m_data['VIXM']).iloc[-1] if 'VIXY' in m_data.columns else 1.0
        market_panic_adj = 0.4 if vix_ratio > 1.05 else 0.0
    except:
        futures_change_pct, vix_ratio, market_panic_adj = 0.0, 1.0, 0.0

    # 參數設定
    entry_mult = 1.0  # 核心要求：建倉維持 1.0 ATR
    base_mults = {'⭐⭐⭐': 1.3, '⭐⭐⭐⭐': 1.7, '⭐⭐⭐⭐⭐': 2.1}

    results = []
    active_items = [c for c in configs if c['symbol'].strip() and c['symbol'] != "代碼"]

    for item in active_items:
        t = item['symbol'].upper()
        frac_tag = "🔹" if item['fractional'] else ""

        try:
            ticker_obj = yf.Ticker(t)
            hist = ticker_obj.history(period="1y")
            if len(hist) < 60: continue

            hist['RSI'] = calculate_rsi(hist['Close'])
            hist['ATR'] = calculate_atr(hist['High'], hist['Low'], hist['Close'])

            today_data = hist.iloc[-1]
            prev_data = hist.iloc[-2]
            y_close = prev_data['Close']
            y_rsi = prev_data['RSI']
            current_atr = hist['ATR'].iloc[-2]

            # 波動補償計算 (僅用於防禦等級)
            avg_atr_long = hist['ATR'].iloc[-60:-2].mean()
            vol_spike = current_atr > (avg_atr_long * 1.3)
            total_panic_adj = market_panic_adj + (0.2 if vol_spike else 0.0)

            # 定錨邏輯
            is_market_open = (today_data.name.date() == datetime.today().date()) and (today_data['Open'] > 0)
            base_ref = today_data['Open'] if is_market_open else min(y_close, y_close * (1 + futures_change_pct/100))
            status_msg = "🟢開盤定錨" if is_market_open else "🟡盤前預判"
            rsi_status = f"⚠️高檔" if y_rsi > 70 else (f"🔥超賣" if y_rsi < 30 else f"中性({y_rsi:.0f})")

            row = {
                '標的': f"{t} {frac_tag}",
                '狀態': f"{status_msg}\n{rsi_status}",
                '波動補償': f"{'⚡加寬' if total_panic_adj > 0 else '一般'}"
            }

            # --- 回測工具 ---
            def get_win_rate(m, days=2):
                wins = []
                # 遍歷過去一年的數據進行模擬掛單
                for i in range(-min(len(hist)-5, 250), -4):
                    d_ref = min(hist.iloc[i-1]['Close'], hist.iloc[i]['Open'])
                    sim_target = d_ref - (hist.iloc[i-1]['ATR'] * m)
                    if hist.iloc[i]['Low'] <= sim_target:
                        # 觸發後，看 N 天後的收盤價是否高於買入價
                        wins.append(1 if hist.iloc[i+days]['Close'] > sim_target else 0)
                return f"{sum(wins)/len(wins)*100:.0f}% ({len(wins)}次)" if wins else "無觸發"

            # --- 1.0 ATR 建倉計算 (固定倍數) ---
            entry_p = base_ref - (current_atr * entry_mult)
            entry_diff = ((entry_p - y_close) / y_close) * 100
            row['💰建倉價格'] = f"<b>${entry_p:.2f}</b> <span style='color:#007bff; font-size:11px'>({entry_diff:+.2f}%)</span>\n<span style='font-size:10px; color:#666'>年勝率: {get_win_rate(entry_mult)}</span>"

            # --- 防禦等級計算 (隨波動加寬) ---
            for label, b_mult in base_mults.items():
                if label == '⭐⭐⭐' and y_rsi > 70:
                    row[label] = "⛔ 風險過高"
                    continue

                f_mult = b_mult + total_panic_adj
                target_p = base_ref - (current_atr * f_mult)
                diff_pct = ((target_p - y_close) / y_close) * 100
                row[label] = f"<b>${target_p:.2f}</b> <span style='color:#007bff; font-size:11px'>({diff_pct:+.2f}%)</span>\n<span style='font-size:10px; color:#666'>年勝率: {get_win_rate(f_mult, 3)}</span>"

            results.append(row)
        except: continue

    return pd.DataFrame(results), futures_change_pct, vix_ratio, market_panic_adj

# --- UI 介面 ---
title_html = widgets.HTML("<h3>🛡️ 絕對防守 V6.3 (建倉 1.0 ATR 版)</h3>")
defaults = ['VOO', 'QQQM', 'VEA', 'FRDM', 'ARKX', 'GOOGL', 'XOVR', 'AAPL', '代碼', '代碼']
input_widgets = []

for i in range(10):
    symbol_text = widgets.Text(value=defaults[i], layout=widgets.Layout(width='80px'))
    frac_check = widgets.Checkbox(value=True, description='碎股', indent=False, layout=widgets.Layout(width='auto'))
    input_widgets.append({'t': symbol_text, 'f': frac_check})

col1 = widgets.VBox([widgets.HBox([w['t'], w['f']]) for w in input_widgets[:5]])
col2 = widgets.VBox([widgets.HBox([w['t'], w['f']]) for w in input_widgets[5:]])
btn_scan = widgets.Button(description="🛡️ 執行全防禦掃描", layout=widgets.Layout(width='100%', height='50px'), button_style='danger')
output = widgets.Output()

def on_click(b):
    with output:
        clear_output()
        print("⏳ 正在計算 1.0 ATR 建倉價與防禦區間... 請稍候")
        configs = [{'symbol': w['t'].value.strip(), 'fractional': w['f'].value} for w in input_widgets]
        df_res, f_pct, v_ratio, m_adj = run_ultimate_v6_3(configs)
        clear_output()
        print(f"期貨盤前: {f_pct:+.2f}% | VIX比值: {v_ratio:.2f} {'🔥(恐慌加成中)' if m_adj > 0 else '☁️'}")

        if not df_res.empty:
            styler = df_res.style.hide(axis='index')
            styler.set_properties(**{'text-align': 'center', 'border': '1px solid #ccc', 'white-space': 'pre-wrap'})
            styler.set_table_styles([
                {'selector': 'th', 'props': [('background-color', '#f4f4f4'), ('color', '#333'), ('text-align', 'center')]},
                {'selector': 'td.col3', 'props': [('background-color', '#e6f7ff')]} # 建倉欄位高亮
            ])
            display(styler)
        else:
            print("❌ 找不到有效數據")

btn_scan.on_click(on_click)
display(widgets.VBox([title_html, widgets.HBox([col1, widgets.Label(layout={'width':'30px'}), col2]), btn_scan, output]))

三星級：【穩健建倉區】

*   量化標準：通常設定在基準價下方約 $1.3x$ ATR（隨 VIX 調升至 $1.7x$）。
*   實戰含義：代表股價出現了「顯著但尚在常態範圍內」的回檔。
*   適用情境：如果您非常看好某標的且尚未持有部位，三星級是為了防止錯過行情而設定的「初步上車點」。
*   預期表現：成交次數相對較多。但正如您在 QQQM 看到的 $0\%$ 勝率，三星級在遇到連續殺盤時，防禦力可能不足，買入後可能仍有短暫浮虧。



四星級：【安全加碼區】

*   量化標準：設定在基準價下方約 $1.7x$ ATR（隨 VIX 調升至 $2.1x$）。
*   實戰含義：代表股價進入了「當日極度超跌」的狀態。這通常是日內空頭動能接近竭盡的信號。
*   適用情境：新買入：如果您堅持「絕對低價」，這就是您的首選掛單位。已有持股：這是最理想的「分批攤平」點，因為在統計上，股價要在一天內跌破四星級的機率很低，通常會有強勁的反彈力道。
*   預期表現：成交次數極少，但 3 日勝率通常最高且最穩健。


五星級：【極端抄底區】

*   量化標準：設定在基準價下方約 $2.1x$ ATR（隨 VIX 調升至 $2.5x$）。
*   實戰含義：代表發生了「黑天鵝」或「非理性恐慌噴發」。這是正常市場波動下幾乎不可能觸及的價格。
*   適用情境：這是一個**「捕獸夾」**。您不期待它天天成交，但您會把這筆資金掛在那裡。如果哪天市場突然閃崩觸及這個價位，您買入的瞬間就已經擁有了極大的安全邊際（Margin of Safety）。
*   預期表現：月成交次數通常為 $0$。一旦成交，這往往是長線股價的「黃金坑」底部。